# ==============================================================================
# 🩺 CLINICAL CARDIOLOGY SCREENING: HEART DISEASE VIA BAGGED KNN CLASSIFIER
# ==============================================================================
### Core Theoretical Principles:
Demonstrates **Non-Tree Bagging**: wrapping an ensemble of K-Nearest Neighbors estimators on random bootstrap subsamples (`max_samples=0.8`) to smooth out local metric boundary noise.


In [1]:
# STEP 1: ENTERPRISE ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.ensemble import BaggingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, recall_score, f1_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Cardiology Environment ready.")


✅ STEP 1: Cardiology Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET HARMONIZATION
Loading Kaggle Heart Disease clinical dataset (1,025 patient records).


In [2]:
# STEP 2 & 3: INGESTION & AUDIT
data_path = 'data/heart_disease/heart.csv'
df = pd.read_csv(data_path)

print(f"📊 Dataset Shape: {df.shape[0]} patients, {df.shape[1]} clinical attributes")
print(f"Target Distribution:\n{df['target'].value_counts(normalize=True).round(3)}")
df.head(3)


📊 Dataset Shape: 920 patients, 14 clinical attributes
Target Distribution:
target
1    0.553
0    0.447
Name: proportion, dtype: float64


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and target `target`.


In [3]:
# STEP 4 & 5: SEGREGATION
X = df.drop(columns=['target']).copy()
y = df['target'].copy()

print(f"Biomarkers: {X.columns.tolist()}")


Biomarkers: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']


In [4]:
# STEP 6: STRATIFIED SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f"Train Cohort: {X_train.shape[0]} patients")
print(f"Test Cohort : {X_test.shape[0]} patients")


Train Cohort: 736 patients
Test Cohort : 184 patients


## ⚙️ STEP 7: PREPROCESSOR PIPELINE
Standardizing numerical vitals and encoding categorical features.


In [5]:
# STEP 7: PREPROCESSOR
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = X.select_dtypes(exclude=[np.number]).columns.tolist()

preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('ohe', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False))
    ]), cat_cols)
])
print("✅ STEP 7: Clinical Preprocessor configured.")


✅ STEP 7: Clinical Preprocessor configured.


## ⚠️ STEP 8: BASELINE PIPELINE FIT
Fitting BaggingClassifier with KNN base estimators.


In [6]:
# STEP 8: BASELINE BAGGED KNN FIT
pipe_base = Pipeline([
    ('prep', preprocessor),
    ('bag', BaggingClassifier(
        estimator=KNeighborsClassifier(n_neighbors=5),
        n_estimators=30,
        max_samples=0.8,
        oob_score=True,
        random_state=42
    ))
])
pipe_base.fit(X_train, y_train)

acc_base = pipe_base.score(X_test, y_test)
print(f"Baseline Bagged KNN Accuracy: {acc_base * 100:.2f}%")
print(f"Bagged KNN OOB Score        : {pipe_base.named_steps['bag'].oob_score_ * 100:.2f}%")


Baseline Bagged KNN Accuracy: 81.52%
Bagged KNN OOB Score        : 79.89%


## 🎯 STEP 9: TUNING HYPERPARAMETERS VIA 5-FOLD CV
Optimizing `n_estimators` and `estimator__n_neighbors`.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'bag__n_estimators': [20, 40],
    'bag__estimator__n_neighbors': [3, 5, 7]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid = GridSearchCV(pipe_base, param_grid=param_grid, cv=cv, scoring='recall', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Clinical Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 5-Fold CV Recall: {grid.best_score_*100:.2f}%")


🏆 Best Clinical Hyperparameters: {'bag__estimator__n_neighbors': 5, 'bag__n_estimators': 40}
🎯 Best 5-Fold CV Recall: 82.79%


## 📊 STEP 10 & 11: CLINICAL EVALUATION REPORT & RESIDUALS
Evaluating clinical sensitivity and ROC-AUC.


In [8]:
# STEP 10 & 11: EVALUATION REPORT
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

print("📋 CLINICAL BAGGED KNN EVALUATION REPORT:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Cardiovascular Risk']))
print(f"Test Recall (Sensitivity) : {recall_score(y_test, y_pred)*100:.2f}%")
print(f"Test ROC-AUC Score        : {roc_auc_score(y_test, y_prob):.4f}")


📋 CLINICAL BAGGED KNN EVALUATION REPORT:
                     precision    recall  f1-score   support

             Normal       0.81      0.77      0.79        82
Cardiovascular Risk       0.82      0.85      0.84       102

           accuracy                           0.82       184
          macro avg       0.81      0.81      0.81       184
       weighted avg       0.81      0.82      0.81       184

Test Recall (Sensitivity) : 85.29%
Test ROC-AUC Score        : 0.8838


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & EMERGENCY TRIAGE SMOKE TEST
Deploying pipeline and validating bagged KNN inference latency.


In [9]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/heart_disease_bagging_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_patient = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_diag = loaded_pipe.predict(sample_patient)[0]
pred_risk = loaded_pipe.predict_proba(sample_patient)[0, 1]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🏥 LIVE BAGGED KNN CLINICAL TRIAGE:")
print(f"   Diagnostic Flag : {'⚠️ CARDIAC RISK' if pred_diag == 1 else '✅ HEALTHY'}")
print(f"   Bagged Risk     : {pred_risk * 100:.2f}%")
print(f"   Triage Latency  : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/heart_disease_bagging_pipeline.joblib (399,424 bytes)



🏥 LIVE BAGGED KNN CLINICAL TRIAGE:
   Diagnostic Flag : ⚠️ CARDIAC RISK
   Bagged Risk     : 61.00%
   Triage Latency  : 128.918 ms (SLA < 2.0ms: CHECK)
